# NER PII/SPII tiếng Việt (XLM-RoBERTa)

Fine-tune **`xlm-roberta-base`** cho bài toán nhận diện thực thể PII/SPII trên dữ liệu hành chính bạn đã làm sạch (text + span theo offset ký tự).


1. **Cửa sổ trượt (sliding window):** văn bản dài (trung bình ~550 token, tối đa ~1.100) sẽ bị cắt nếu để `max_length=512`. Ta cắt thành nhiều đoạn chồng nhau (`stride`) để **không mất thực thể ở đuôi**.
2. **Span lồng nhau → phẳng hoá (innermost-wins):** token-classification chuẩn chỉ gán **1 nhãn/token**. Khi span lồng nhau (vd `full_name` nằm trong `family_relations`), token thuộc về span **ngắn nhất** (cụ thể nhất).

> Chạy trên **GPU** (Colab: Runtime → Change runtime type → T4 GPU). Có thể chạy thử ngay trên file mẫu 10 dòng để kiểm tra pipeline, rồi trỏ `DATA_GLOB` sang 15k để huấn luyện thật.

## 1. Cài thư viện

In [1]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (sẽ chậm — bật GPU)")

torch 2.7.1+cu118 | CUDA: True | NVIDIA GeForce RTX 3070 Ti


## 2. Cấu hình

Trỏ `DATA_GLOB` tới file dữ liệu đã làm sạch. Nhận **một file** (`"data.json"`) hoặc **nhiều shard** (`"fixed/*.json"`). Mỗi file là một list các record `{text, spans:[{start,end,field_name,label}]}`.

In [2]:
DATA_GLOB    = "data.json"          # <-- đổi sang file/shard 15k của bạn, vd "fixed/*.fixed.json"
MODEL_NAME   = "xlm-roberta-base"    # hoặc "distilbert-base-multilingual-cased" (nhẹ/nhanh hơn)
OUTPUT_DIR   = "ner-pii-vi"

MAX_LEN      = 384                   # độ dài mỗi đoạn (token)
STRIDE       = 128                   # phần chồng giữa các đoạn (sliding window)
EPOCHS       = 3
LR           = 2e-5
TRAIN_BS     = 8                     # tăng lên 16/32 nếu GPU khoẻ
EVAL_BS      = 16
VAL_RATIO    = 0.1
SEED         = 42

## 3. Nạp dữ liệu & thống kê

In [3]:
import glob, json, random
random.seed(SEED)

def load_records(pattern):
    files = sorted(glob.glob(pattern))
    assert files, f"Không thấy file nào khớp {pattern!r}"
    recs = []
    for f in files:
        data = json.load(open(f, encoding="utf-8"))
        recs.extend(data if isinstance(data, list) else [data])
    return recs

records = load_records(DATA_GLOB)

# chỉ giữ trường cần thiết, và lọc span hợp lệ (offset khớp text)
clean = []
for r in records:
    text = r.get("text", "")
    starts, ends, fields = [], [], []
    for s in r.get("spans", []):
        a, b, fld = s.get("start"), s.get("end"), s.get("field_name")
        if a is None or b is None or not fld or a >= b or b > len(text):
            continue
        starts.append(a); ends.append(b); fields.append(fld)
    if text and fields:
        clean.append({"text": text, "starts": starts, "ends": ends, "fields": fields})

print(f"{len(clean)} record dùng được (trên {len(records)} đã nạp)")
from collections import Counter
fc = Counter(f for r in clean for f in r["fields"])
print(f"{len(fc)} loại thực thể | tổng {sum(fc.values())} span")
print("Phân bố:", dict(fc.most_common()))

8410 record dùng được (trên 8410 đã nạp)
24 loại thực thể | tổng 151607 span
Phân bố: {'full_name': 21211, 'dob': 9283, 'address': 8711, 'cccd': 8484, 'marital_status': 8377, 'family_relations': 8314, 'phone': 8087, 'email': 7431, 'passport_number': 6918, 'driver_license': 6241, 'vehicle_plate': 6199, 'digital_account': 6104, 'bank_account': 4446, 'health_status': 4196, 'ethnicity': 3925, 'location_data': 3798, 'biometric': 3773, 'religion': 3771, 'sexual_orientation': 3765, 'behavioral_data': 3756, 'political_view': 3750, 'private_life': 3711, 'criminal_record': 3682, 'eid_credentials': 3674}


## 4. Bộ nhãn BIO

Mỗi loại trường `X` → `B-X` (token mở đầu) và `I-X` (token tiếp theo); token không thuộc thực thể nào → `O`.

In [4]:
fields_sorted = sorted({f for r in clean for f in r["fields"]})
label_list = ["O"] + [f"{p}-{f}" for f in fields_sorted for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print(len(label_list), "nhãn:", label_list)

49 nhãn: ['O', 'B-address', 'I-address', 'B-bank_account', 'I-bank_account', 'B-behavioral_data', 'I-behavioral_data', 'B-biometric', 'I-biometric', 'B-cccd', 'I-cccd', 'B-criminal_record', 'I-criminal_record', 'B-digital_account', 'I-digital_account', 'B-dob', 'I-dob', 'B-driver_license', 'I-driver_license', 'B-eid_credentials', 'I-eid_credentials', 'B-email', 'I-email', 'B-ethnicity', 'I-ethnicity', 'B-family_relations', 'I-family_relations', 'B-full_name', 'I-full_name', 'B-health_status', 'I-health_status', 'B-location_data', 'I-location_data', 'B-marital_status', 'I-marital_status', 'B-passport_number', 'I-passport_number', 'B-phone', 'I-phone', 'B-political_view', 'I-political_view', 'B-private_life', 'I-private_life', 'B-religion', 'I-religion', 'B-sexual_orientation', 'I-sexual_orientation', 'B-vehicle_plate', 'I-vehicle_plate']


## 5. Tách token + gắn nhãn (cửa sổ trượt + phẳng hoá span lồng)

`_offsets_to_bio` là phần lõi: với mỗi token (có offset ký tự `[cs, ce)`), tìm span **ngắn nhất** bao trọn token rồi gán `B-/I-`. Token đặc biệt (`[CLS]`, `[SEP]`, padding) → `-100` (bỏ qua khi tính loss).

> ⚡ Token hoá **trực tiếp theo lô** (không dùng `datasets.map`) để tránh treo.

In [5]:
import os, time
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")   # batch-encode đa luồng (nhanh); KHÔNG fork -> không deadlock

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "Cần fast tokenizer để có offset_mapping."

def _offsets_to_bio(offsets, starts, ends, fields):
    """offsets: list (cs,ce) theo text gốc. Trả list id nhãn cùng độ dài."""
    order = sorted(range(len(fields)), key=lambda i: ends[i] - starts[i])   # span ngắn -> dài
    out, prev = [], None
    for cs, ce in offsets:
        if cs == ce:                                   # token đặc biệt
            out.append(-100); prev = None; continue
        hit = None
        for i in order:                                # span ngắn nhất bao trọn token
            if starts[i] <= cs and ce <= ends[i]:
                hit = i; break
        if hit is None:
            out.append(label2id["O"]); prev = None
        else:
            out.append(label2id[("I-" if prev == hit else "B-") + fields[hit]]); prev = hit
    return out

def encode_records(records, enc_batch=2000):
    """Token hoá TRỰC TIẾP theo lô (KHÔNG dùng datasets.map -> tránh treo) + gắn nhãn.
    Trả dict cột {input_ids, attention_mask, labels} để dựng Dataset."""
    ids, mask, labels = [], [], []
    t0 = time.time()
    for s in range(0, len(records), enc_batch):
        part = records[s:s + enc_batch]
        enc = tokenizer([r["text"] for r in part], truncation=True, max_length=MAX_LEN,
                        stride=STRIDE, return_overflowing_tokens=True,
                        return_offsets_mapping=True, padding=False)
        smap = enc["overflow_to_sample_mapping"]
        for k, offs in enumerate(enc["offset_mapping"]):
            r = part[smap[k]]
            ids.append(enc["input_ids"][k]); mask.append(enc["attention_mask"][k])
            labels.append(_offsets_to_bio(offs, r["starts"], r["ends"], r["fields"]))
        print(f"  ...{min(s + enc_batch, len(records))}/{len(records)} record -> {len(ids)} đoạn ({time.time() - t0:.0f}s)")
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}

D:\workspace\project\VKU\DACN1\fixer\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## 6. Tạo Dataset và chia train/val

In [6]:
from datasets import Dataset

random.shuffle(clean)
n_val = max(1, int(len(clean) * VAL_RATIO))
val_raw, train_raw = clean[:n_val], clean[n_val:]
print(f"train {len(train_raw)} | val {len(val_raw)} record")

print("Token hoá train...");  train_ds = Dataset.from_dict(encode_records(train_raw))
print("Token hoá val...");    val_ds   = Dataset.from_dict(encode_records(val_raw))
print(f"Sau cửa sổ trượt: train {len(train_ds)} đoạn | val {len(val_ds)} đoạn")

train 7569 | val 841 record
Token hoá train...
  ...2000/7569 record -> 4214 đoạn (2s)
  ...4000/7569 record -> 8517 đoạn (4s)
  ...6000/7569 record -> 12818 đoạn (7s)
  ...7569/7569 record -> 16137 đoạn (8s)
Token hoá val...
  ...841/841 record -> 1819 đoạn (1s)
Sau cửa sổ trượt: train 16137 đoạn | val 1819 đoạn


## 7. Mô hình + chỉ số (seqeval, mức thực thể)

In [7]:
import numpy as np
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)

collator = DataCollatorForTokenClassification(tokenizer)

def decode(preds, labels):
    out_p, out_l = [], []
    for pr, la in zip(preds, labels):
        tp, tl = [], []
        for p_i, l_i in zip(pr, la):
            if l_i != -100:
                tp.append(id2label[int(p_i)]); tl.append(id2label[int(l_i)])
        out_p.append(tp); out_l.append(tl)
    return out_p, out_l

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    p, l = decode(preds, labels)
    return {"precision": precision_score(l, p),
            "recall":    recall_score(l, p),
            "f1":        f1_score(l, p)}

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 8. Huấn luyện

In [8]:
import inspect
from transformers import TrainingArguments, Trainer

ta = dict(
    output_dir=OUTPUT_DIR, learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    weight_decay=0.01, warmup_ratio=0.06, logging_steps=50,
    fp16=torch.cuda.is_available(), seed=SEED,
    load_best_model_at_end=True, metric_for_best_model="f1",
    greater_is_better=True, report_to="none",
)
# tên tham số khác nhau giữa các phiên bản transformers
_par = inspect.signature(TrainingArguments.__init__).parameters
strat = "eval_strategy" if "eval_strategy" in _par else "evaluation_strategy"
ta[strat] = "epoch"; ta["save_strategy"] = "epoch"; ta["save_total_limit"] = 2

trainer_kwargs = dict(model=model, args=TrainingArguments(**ta),
                      train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=collator, compute_metrics=compute_metrics)
# transformers mới đổi tên tham số tokenizer -> processing_class
_tp = inspect.signature(Trainer.__init__).parameters
trainer_kwargs["processing_class" if "processing_class" in _tp else "tokenizer"] = tokenizer
trainer = Trainer(**trainer_kwargs)
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.055000,0.052458,0.940318,0.967541,0.953735
2,0.046600,0.042531,0.943702,0.973165,0.958207
3,0.027200,0.038980,0.952954,0.973165,0.962954


TrainOutput(global_step=6054, training_loss=0.10953274329365541, metrics={'train_runtime': 1427.0622, 'train_samples_per_second': 33.924, 'train_steps_per_second': 4.242, 'total_flos': 9490451100030726.0, 'train_loss': 0.10953274329365541, 'epoch': 3.0})

## 9. Đánh giá chi tiết theo từng loại thực thể

In [9]:
pred = trainer.predict(val_ds)
preds = np.argmax(pred.predictions, axis=2)
p, l = decode(preds, pred.label_ids)
print(classification_report(l, p, digits=3))

                    precision    recall  f1-score   support

           address      0.972     0.978     0.975       998
      bank_account      0.950     0.961     0.956       539
   behavioral_data      0.954     0.987     0.970       530
         biometric      0.974     0.992     0.983       484
              cccd      0.977     0.978     0.978      1099
   criminal_record      0.970     0.990     0.980       492
   digital_account      0.978     0.984     0.981       821
               dob      0.979     0.988     0.983      1113
    driver_license      0.948     0.969     0.959       812
   eid_credentials      0.982     0.982     0.982       488
             email      0.960     0.974     0.967       946
         ethnicity      0.943     0.939     0.941       473
  family_relations      0.920     0.966     0.943      1126
         full_name      0.990     0.995     0.992      2478
     health_status      0.795     0.916     0.851       571
     location_data      0.869     0.911

## 10. Lưu mô hình

In [10]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Đã lưu vào", OUTPUT_DIR)

Đã lưu vào ner-pii-vi


## 11. Thử suy luận (demo)

Gộp các token con (`B-`/`I-`) thành span theo offset ký tự và trích thực thể từ một đoạn văn bản mới.

In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

if 'model' not in globals() or 'tokenizer' not in globals():
    tokenizer = AutoTokenizer.from_pretrained("./ner-pii-vi")
    model = AutoModelForTokenClassification.from_pretrained("./ner-pii-vi")
id2label = model.config.id2label

# HÀM DỰ ĐOÁN FULL VĂN BẢN (CHUNKING)
def predict_full_document(full_text, model=model, tokenizer=tokenizer):
    model.eval()

    # 1. Chẻ văn bản theo dấu xuống dòng
    paragraphs = []
    start_idx = 0
    for p in full_text.split('\n'):
        if p.strip():
            paragraphs.append((start_idx, p))
        start_idx += len(p) + 1

    all_entities = []
    with torch.no_grad():
        for p_offset, chunk_text in paragraphs:
            # 2. Quét từng đoạn ngắn
            enc = tokenizer(chunk_text, return_offsets_mapping=True, truncation=True,
                            max_length=384, return_tensors="pt")
            offsets = enc.pop("offset_mapping")[0].tolist()

            logits = model(**{k: v.to(model.device) for k, v in enc.items()}).logits[0]
            ids = logits.argmax(-1).tolist()

            cur = None
            for (cs, ce), pid in zip(offsets, ids):
                if cs == ce: continue
                lab = id2label[int(pid)]
                if lab == "O":
                    if cur: all_entities.append(cur); cur = None
                    continue
                tag, fld = lab.split("-", 1)
                if cur and tag == "I" and cur["field_name"] == fld:
                    cur["end"] = ce + p_offset
                else:
                    if cur: all_entities.append(cur)
                    cur = {"field_name": fld, "start": cs + p_offset, "end": ce + p_offset}
            if cur: all_entities.append(cur)

    for e in all_entities:
        e["text"] = full_text[e["start"]:e["end"]]
    return all_entities

# ==========================================
sample_data = val_raw[2]
demo_text = sample_data["text"]

print(demo_text, "\n")

print("="*60)

print("NHÃN CÓ SẴN TRONG DỮ LIỆU:")
try:
    for st, en, fld in zip(sample_data["starts"], sample_data["ends"], sample_data["fields"]):
        print(f"  [{fld:18}] {demo_text[st:en]!r}")
except KeyError:
    pass

print("-" * 60)

print("MÔ HÌNH DỰ ĐOÁN:")
for e in predict_full_document(demo_text):
    print(f"  [{e['field_name']:18}] {e['text']!r}")

Tôi nhận được yêu cầu hỗ trợ trường học của Giáo viên THCS Phạm Đình Khoa sinh năm 1994. Ông đang cần bổ sung thông tin cho thủ tục Thủ tục công nhận hoạt động đại lý làm thủ tục hải quan tại Bộ Tài chính.

Để hoàn thành hồ sơ, tôi cần thu thập thông tin cá nhân của ông. Ông có hộ khẩu thường trú tại Phường/Xã Nhân Chính, Quận/Huyện Nam Từ Liêm, Tỉnh/Thành phố Hà Nội. Ông có số tài khoản ngân hàng 190505501418 và số giấy phép lái xe GPLX258330748.

Ông cũng cần cung cấp thông tin liên hệ khẩn cấp, trong đó có số điện thoại 0815216402 và số căn cước công dân 001443151201. Ngoài ra, ông cần xác nhận rằng ông không khai báo quan điểm chính trị trong hồ sơ và không có lịch sử truy cập cổng dịch vụ công vào buổi tối.

Tôi cũng cần thông tin về vị trí thường trú của ông, trong đó có mã 581410. Ông cần cung cấp thông tin về người liên hệ khẩn cấp, trong đó có thân nhân của mình. Cuối cùng, ông cần xác nhận rằng ông có tình trạng hôn nhân là Độc thân và không có quan hệ gia đình khác.

Tôi sẽ 